# Exercise 06 — Index models for portfolio construction

MSc Finance · Investments · FHNW · Autumn 2026

One regression, run twenty times, produces every input the index model needs. This
notebook estimates it, asks how much of the estimate is worth carrying forward, and then
spends the answer on a core-satellite portfolio — which promptly loses to the core.

Data: twenty SPI constituents, monthly total returns in CHF, December 2004 to December
2025, with the SARON as the risk-free rate. Source: S&P Capital IQ and the
[Swiss National Bank data portal](https://data.snb.ch). The file carries no index series,
so the equally weighted portfolio of the twenty stocks is our market proxy — both sides
of every regression then carry the same survivorship bias.

Run **Runtime → Restart and run all** before you trust any number in here.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from matplotlib.ticker import PercentFormatter
from scipy.optimize import minimize

from exercise_utils import FHNW, ASSET_CYCLE, setup_style, load_returns, save_results
setup_style()

BASE = "https://raw.githubusercontent.com/KroeTiA/Investments/main/"
DATA_URL = BASE + "Exercise_06/data/swiss_equities_monthly.xlsx"

STOCK = "SWX:UBSG"

# Four non-overlapping five-year windows, plus the period the backtest evaluates.
BLOCKS = {"2005-2009": slice("2005-01", "2009-12"),
          "2010-2014": slice("2010-01", "2014-12"),
          "2015-2019": slice("2015-01", "2019-12"),
          "2020-2024": slice("2020-01", "2024-12")}
W1 = BLOCKS["2005-2009"]
OOS = slice("2010-01", "2025-12")

print("Setup complete.")

## The data

Three steps, all visible: index levels to returns, the money-market rate from percent per
annum to a monthly decimal, and excess returns on both sides. The market proxy is the
equally weighted portfolio of the twenty stocks.

In [ ]:
# Monthly total-return index levels -> monthly returns.
rets = load_returns(DATA_URL, sheet="Prices", index_col="Date", to_returns=True)

# SARON, quoted in percent per annum. Compound to a monthly rate; do not divide by 12.
saron = pd.read_excel(DATA_URL, sheet_name="RiskFree", index_col="Date")["Rate_pa_pct"]
rf = ((1 + saron / 100) ** (1 / 12) - 1).reindex(rets.index)

# The Info sheet is a two-column field/value list; the ticker rows are the tail of it.
info = pd.read_excel(DATA_URL, sheet_name="Info")
tick = info[info["Field"].astype(str).str.startswith("SWX:")]
NAME = dict(zip(tick["Field"], tick["Value"]))

STOCKS = list(rets.columns)
mkt = rets.mean(axis=1)                 # equally weighted market proxy
rx = rets.sub(rf, axis=0)               # excess returns, stocks
mkt_x = mkt - rf                        # excess returns, market proxy


def sharpe(r):
    """Annualised Sharpe ratio of a monthly excess-return series."""
    return 12 * r.mean() / (np.sqrt(12) * r.std())


print(f"{len(rets)} monthly returns, {len(STOCKS)} stocks, "
      f"{rets.index.min():%b %Y} to {rets.index.max():%b %Y}")
print(f"market proxy over {W1.start}-{W1.stop}:  "
      f"E(R_M^e) {12 * mkt_x.loc[W1].mean():.2%} p.a., "
      f"sigma_M {np.sqrt(12) * mkt_x.loc[W1].std():.2%} p.a.")

## Task 1 — One stock, one window, four numbers

UBS Group against the market proxy, over the first five years of the sample: January 2005
to December 2009. That is the same 60-month window the backtest at the end of this
notebook starts from, and it is the window length the lecture used throughout.

Estimate the security characteristic line

$$R_{i,t}^{e} \;=\; \alpha_i \;+\; \beta_i R_{M,t}^{e} \;+\; e_{i,t}$$

and report all four numbers it delivers: $\alpha$ (annualised), $\beta$, $R^2$ and
$\sigma(e)$ — each with its standard error where it has one. Test $\beta$ against 1, not
against 0: the interesting question is whether the stock is more cyclical than the market,
not whether it moves with it at all.

Then split the stock's variance into the two parts the lecture separated,

$$\sigma_i^2 \;=\; \beta_i^2 \sigma_M^2 \;+\; \sigma^2(e_i),$$

and confirm that they add up. Finally draw the line: one point per month, the fit through
them, and the intercept marked.

**Deliverable.** The four parameters with their t-statistics, the variance split, and one
figure. Then answer in one sentence: which of the four numbers would you be willing to put
into a portfolio decision?

In [ ]:
X = sm.add_constant(mkt_x.loc[W1])
# TODO: regress the stock's excess return on the market's over W1, and call the fit `res`
# TODO: annualise alpha and sigma(e); beta and R2 need no annualisation
# TODO: the two halves of the variance, both annualised

In [ ]:
print(f"{NAME[STOCK]}  vs the market proxy,  {len(res.resid)} months\n")
print(f"  alpha      {alpha:8.2%}   se {se_alpha:6.2%}   t {alpha / se_alpha:6.2f}")
print(f"  beta       {beta:8.3f}   se {se_beta:6.3f}   t vs 1 {(beta - 1) / se_beta:5.2f}")
print(f"  R2         {r2:8.3f}")
print(f"  sigma(e)   {sde:8.2%}   (annualised)")
print(f"  sigma_i    {sd_i:8.2%}   sigma_M {sd_m:.2%}\n")
print(f"  variance   {systematic:.4f} systematic  +  {specific:.4f} firm-specific"
      f"  =  {systematic + specific:.4f}   (sample variance {sd_i ** 2:.4f})")
print(f"  shares     {systematic / (systematic + specific):.1%} systematic / "
      f"{specific / (systematic + specific):.1%} firm-specific   (= R2, up to the "
      f"degrees-of-freedom correction in sigma(e))")

In [ ]:
fig1, ax = plt.subplots(figsize=(6.4, 5.2))
x = mkt_x.loc[W1]
y = rx.loc[W1, STOCK]
grid = np.linspace(x.min(), x.max(), 50)
# TODO: scatter the 60 monthly pairs, then draw the fitted line over `grid`
ax.axhline(0, lw=0.6, color="grey")
ax.axvline(0, lw=0.6, color="grey")
# TODO: mark the intercept - the fitted value when the market's excess return is zero
ax.set_xlabel("Market proxy, monthly excess return")
ax.set_ylabel(f"{NAME[STOCK]}, monthly excess return")
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set_title(f"slope = {beta:.2f},  R2 = {r2:.2f},  sigma(e) = {sde:.0%} p.a.", fontsize=10)
plt.show()

## Task 2 — How much of an estimate survives into the next five years

An estimate is only useful if it says something about the future. Cut the sample into the
four non-overlapping five-year blocks defined in the setup cell and estimate α and β for
all twenty stocks in each of them. Then pool the three consecutive (block, next block)
pairs — sixty observations in total — and regress the next block's estimate on the
previous one:

$$\hat{\beta}_{\text{next}} \;=\; a \;+\; c\,\hat{\beta}_{\text{prev}}
\;+\; u.$$

The slope $c$ is the share of today's estimate that is worth carrying forward; $1 - c$ is
the share that belongs to the prior. Run it for β and, separately, for α.

Then check the slope against what it is supposed to do. Forecast the next block with the
rule *keep k of the estimate, put 1 − k on the prior* — prior 1 for β, prior 0 for α — over
a grid of k, and find the k that minimises the root mean squared forecast error.

**Deliverable.** Two slopes, the loss-minimising k for each, and one figure with the two
scatters. Then answer: why is the number smaller for α than for β?

*Write the estimation as a function `index_model(block)`; Task 3 uses it again.*

In [ ]:
def index_model(block):
    """alpha (annualised), beta and sigma(e) (annualised) for every stock."""
    # TODO: one regression per stock over `block`; return a DataFrame indexed by ticker

In [ ]:
tables = {label: index_model(blk) for label, blk in BLOCKS.items()}
beta_blk = pd.DataFrame({k: t["beta"] for k, t in tables.items()})
alpha_blk = pd.DataFrame({k: t["alpha"] for k, t in tables.items()})

print("beta, by block\n")
print(beta_blk.sort_values("2005-2009").to_string(float_format="{:6.2f}".format))
print(f"\ncross-sectional sd of beta: "
      + "  ".join(f"{k} {v:.2f}" for k, v in beta_blk.std().items()))
print(f"\nalpha (annualised), {NAME[STOCK]}:  "
      + "  ".join(f"{k} {v:+.1%}" for k, v in alpha_blk.loc[STOCK].items()))

In [ ]:
def persistence(panel):
    """Pool the three consecutive block pairs and regress next on previous."""
    prev = np.concatenate([panel.iloc[:, k].values for k in range(3)])
    nxt = np.concatenate([panel.iloc[:, k + 1].values for k in range(3)])
    # TODO: regress nxt on prev; return the fitted object and the two vectors

In [ ]:
fit_b, prev_b, next_b = persistence(beta_blk)
fit_a, prev_a, next_a = persistence(alpha_blk)
KS = np.linspace(0, 1, 101)
# TODO: RMSE of "keep k of the estimate, 1-k on the prior" - prior 1 for beta, 0 for alpha

In [ ]:
for lbl, fit, rmse in [("beta ", fit_b, rmse_b), ("alpha", fit_a, rmse_a)]:
    k_best = KS[int(np.argmin(rmse))]
    print(f"{lbl}:  slope {fit.params[1]:.3f}  (se {fit.bse[1]:.3f}, "
          f"t {fit.tvalues[1]:5.2f})   R2 {fit.rsquared:.3f}"
          f"   |   best k {k_best:.2f}   RMSE {min(rmse):.4f} "
          f"vs {rmse[-1]:.4f} unshrunk")

SHRINK_BETA = 2 / 3
SHRINK_ALPHA = 1 / 3
print(f"\nrules of thumb we carry into Task 3:  "
      f"beta {SHRINK_BETA:.2f} x estimate + {1 - SHRINK_BETA:.2f} x 1,   "
      f"alpha {SHRINK_ALPHA:.2f} x estimate")

In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(10.4, 4.6))
panels = [("beta", prev_b, next_b, fit_b, 1.0, False),
          ("alpha (annualised)", prev_a, next_a, fit_a, 0.0, True)]
for ax, (lbl, prev, nxt, fit, prior, pct) in zip(axes, panels):
    g = np.linspace(prev.min(), prev.max(), 50)
    # TODO: scatter the pairs, add the 45-degree line and the fitted line
    ax.axhline(prior, lw=0.7, color=FHNW["red"])
    ax.set_xlabel(f"{lbl}, five-year block")
    ax.set_ylabel(f"{lbl}, next block")
    if pct:
        ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.legend(fontsize=9, loc="upper left")
fig2.tight_layout()
plt.show()

## Task 3 — Build the satellite, then test it

Put the machinery to work. Treat the 2005–2009 estimates as an analyst's forecasts, pick
the five stocks with the highest α, and build the active sleeve exactly as the lecture
derived it:

$$w_i \;=\; \frac{\alpha_i / \sigma^2(e_i)}{\sum_j \alpha_j / \sigma^2(e_j)},
\qquad
w_A^{0} \;=\; \frac{\alpha_A / \sigma^2(e_A)}{E(R_M^{e}) / \sigma_M^2},
\qquad
w_A^{*} \;=\; \frac{w_A^{0}}{1 + (1 - \beta_A) w_A^{0}}.$$

Report what the construction promises through $SR_P^2 = SR_M^2 + IR_A^2$. Do it twice:
once with the raw estimates, once with the two shrinkage factors from Task 2.

Then stop optimising and start living with it. Hold both portfolios unchanged from January
2010 to December 2025 — sixteen years, no rebalancing, no new forecasts — and report the
Sharpe ratio each actually delivered, next to the core on its own.

**Deliverable.** Two sleeves with their promised Sharpe ratios, the same two portfolios'
realised Sharpe ratios, and one figure. Then answer: the raw sleeve promised an
information ratio above 1. Where did it go?

In [ ]:
def satellite(est, picks, ka=1.0, kb=1.0):
    """Treynor-Black sleeve; ka shrinks alpha, kb shrinks beta toward 1."""
    a, b, se = ka * est.loc[picks, "alpha"], kb * est.loc[picks, "beta"] + (1 - kb), est.loc[picks, "sde"]
    # TODO: weights inside the sleeve, proportional to alpha_i / sigma^2(e_i)
    # TODO: the sleeve's own alpha, beta, residual risk and information ratio
    # TODO: the unadjusted sleeve size w0_A, then the beta adjustment

In [ ]:
est1 = index_model(W1)
er_m = 12 * mkt_x.loc[W1].mean()
sd_m_1 = np.sqrt(12) * mkt_x.loc[W1].std()
sr_m = er_m / sd_m_1

PICKS = list(est1.sort_values("alpha", ascending=False).head(5).index)
print("the analyst's five names:  " + ",  ".join(f"{NAME[p]}" for p in PICKS))
print(est1.loc[PICKS].to_string(float_format="{:8.3f}".format))

w_raw, s_raw = satellite(est1, PICKS)
w_shr, s_shr = satellite(est1, PICKS, ka=SHRINK_ALPHA, kb=SHRINK_BETA)

print(f"\ncore:  E(R_M^e) {er_m:.2%}   sigma_M {sd_m_1:.2%}   SR_M {sr_m:.3f}\n")
print(pd.DataFrame({"raw estimates": s_raw, "shrunk": s_shr})
      .to_string(float_format="{:8.3f}".format))
print("\nweights inside the sleeve\n"
      + pd.DataFrame({"raw": w_raw, "shrunk": w_shr}).to_string(float_format="{:6.3f}".format))

In [ ]:
# TODO: the sleeve's own excess return - the five stocks at the weights you just found
# TODO: the risky portfolio - w*_A in the sleeve, the rest in the core

In [ ]:
rows = {"core (1/N)": mkt_x, "core-satellite, raw": p_raw,
        "core-satellite, shrunk": p_shr}
tbl = pd.DataFrame({k: {"SR 2005-2009": sharpe(v.loc[W1]),
                        "SR 2010-2025": sharpe(v.loc[OOS]),
                        "E(R^e) 2010-2025": 12 * v.loc[OOS].mean(),
                        "sigma 2010-2025": np.sqrt(12) * v.loc[OOS].std()}
                    for k, v in rows.items()}).T
print(tbl.to_string(formatters={"SR 2005-2009": "{:8.3f}".format,
                                "SR 2010-2025": "{:8.3f}".format,
                                "E(R^e) 2010-2025": "{:8.2%}".format,
                                "sigma 2010-2025": "{:8.2%}".format}))
print(f"\npromised by the formula:  raw {s_raw['SR_P']:.3f},  "
      f"shrunk {s_shr['SR_P']:.3f}")

oos = sm.OLS(act_raw.loc[OOS], sm.add_constant(mkt_x.loc[OOS])).fit()
print(f"\nthe raw sleeve out of sample:  alpha {12 * oos.params.iloc[0]:+.2%} p.a. "
      f"(t {oos.tvalues.iloc[0]:.2f}),  IR {12 * oos.params.iloc[0] / np.sqrt(12 * oos.mse_resid):+.3f}"
      f"   -- in sample it was {s_raw['IR_A']:.3f}")

In [ ]:
fig3, ax = plt.subplots(figsize=(8.4, 4.6))
# TODO: cumulative excess return of the three portfolios over the out-of-sample period
ax.axhline(1, lw=0.6, color="grey")
ax.set_xlabel("")
ax.set_ylabel("Cumulative excess return, 1 CHF invested Jan 2010")
ax.legend(loc="upper left")
plt.show()

## Appendix — 1/N, mean-variance and the index model, out of sample

Worked through in full; there is no task here. The cells below run a rolling backtest over
the whole sample: estimate on the most recent 60 months, rebalance every January, hold, and
repeat, from January 2010 to December 2025. Six portfolios of the same twenty stocks,
long-only, fully invested:

1. **1/N** — equal weights, no estimation at all.
2. **Minimum variance, sample covariance** — 210 covariances estimated from 60 months.
3. **Minimum variance, index-model covariance** — the same matrix rebuilt as
   $\Sigma = \beta \beta' \sigma_M^2 + \mathrm{diag}(\sigma^2(e))$, so 41 numbers
   instead of 210.
4. **Tangency, sample moments** — the full Markowitz problem, means included.
5. **Tangency, index model with α ≡ 0** — expected returns forced onto the security market
   line, $E(R_i^e) = \beta_i E(R_M^e)$.
6. **Treynor-Black** — the same index-model covariance, but with the estimated alphas left
   in: $E(R_i^e) = \hat{\alpha}_i + \beta_i E(R_M^e)$.

Rows 5 and 6 differ in one thing only: whether the optimiser is allowed to believe the
estimated alphas.

In [ ]:
WINDOW = 60


def fit_panel(sub, m):
    """Alphas, betas and residual variances for a block of monthly excess returns."""
    X = sm.add_constant(m)
    a = np.zeros(len(STOCKS)); b = np.zeros(len(STOCKS)); v = np.zeros(len(STOCKS))
    for i, c in enumerate(sub.columns):
        f = sm.OLS(sub[c], X).fit()
        a[i], b[i], v[i] = f.params.iloc[0], f.params.iloc[1], f.mse_resid
    return a, b, v


def solve(objective, n):
    """Long-only, fully invested, minimising `objective`."""
    return minimize(objective, np.ones(n) / n, method="SLSQP",
                    bounds=[(0, 1)] * n,
                    constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                    options={"ftol": 1e-12, "maxiter": 2000}).x


def backtest(window=WINDOW):
    n = len(STOCKS)
    keys = ["1/N", "Min-var, sample", "Min-var, index model",
            "Tangency, sample", "Tangency, index (alpha=0)", "Treynor-Black (raw alpha)"]
    out = {k: [] for k in keys}
    dates, w = [], {}
    for t in range(window, len(rx)):
        if (t - window) % 12 == 0:                       # rebalance once a year
            sub, m = rx.iloc[t - window:t], mkt_x.iloc[t - window:t]
            S = np.cov(sub.values, rowvar=False)
            a, b, v = fit_panel(sub, m)
            Si = np.outer(b, b) * m.var() + np.diag(v)
            mu, mu_i, mu_tb = sub.mean().values, b * m.mean(), a + b * m.mean()
            w["1/N"] = np.ones(n) / n
            w["Min-var, sample"] = solve(lambda x: x @ S @ x, n)
            w["Min-var, index model"] = solve(lambda x: x @ Si @ x, n)
            w["Tangency, sample"] = solve(lambda x: -(x @ mu) / np.sqrt(x @ S @ x), n)
            w["Tangency, index (alpha=0)"] = solve(lambda x: -(x @ mu_i) / np.sqrt(x @ Si @ x), n)
            w["Treynor-Black (raw alpha)"] = solve(lambda x: -(x @ mu_tb) / np.sqrt(x @ Si @ x), n)
        r = rx.iloc[t].values
        for k in keys:
            out[k].append(float(w[k] @ r))
        dates.append(rx.index[t])
    return pd.DataFrame(out, index=dates)


paths = backtest()
summary = pd.DataFrame({"E(R^e) p.a.": 12 * paths.mean(),
                        "sigma p.a.": np.sqrt(12) * paths.std(),
                        "Sharpe ratio": paths.apply(sharpe)})
print(f"out of sample {paths.index.min():%b %Y} to {paths.index.max():%b %Y}, "
      f"{len(paths)} months, annual rebalancing\n")
print(summary.to_string(formatters={"E(R^e) p.a.": "{:8.2%}".format,
                                    "sigma p.a.": "{:8.2%}".format,
                                    "Sharpe ratio": "{:8.3f}".format}))

In [ ]:
STYLE = {"1/N": (FHNW["navy"], 2.2),
         "Tangency, index (alpha=0)": (FHNW["green"], 2.0),
         "Treynor-Black (raw alpha)": (FHNW["red"], 1.4),
         "Tangency, sample": (FHNW["orange"], 1.2),
         "Min-var, sample": ("#8A8A8A", 1.2),
         "Min-var, index model": ("#BFBFBF", 1.2)}

fig4, ax = plt.subplots(figsize=(8.6, 4.8))
for lbl, series in paths.items():
    colour, lw = STYLE[lbl]
    ax.plot((1 + series).cumprod(), label=lbl, color=colour, lw=lw)
ax.axhline(1, lw=0.6, color="grey")
ax.set_ylabel("Cumulative excess return, 1 CHF invested Jan 2010")
ax.legend(fontsize=9, loc="upper left")
plt.show()

Three things to take from the table.

**1/N is hard to beat.** It estimates nothing, and it finishes ahead of every strategy that
estimates expected returns from past averages. Both minimum-variance portfolios do what
they promise — they cut volatility by roughly two percentage points — but they pay for it
in return, because the low-variance stocks in this panel were also the lower-returning
ones.

**The index model helps by forbidding, not by improving.** The only portfolio that beats
1/N is the one whose expected returns are forced onto the security market line with α ≡ 0.
Give the same optimiser, with the same covariance matrix, the estimated alphas instead, and
the Sharpe ratio falls by a quarter, from 0.90 to 0.68. The structure was never mainly a better covariance
matrix; it is a rule about what you are allowed to claim to know.

**Twenty stocks is a gentle test.** With n = 20 and 60 months, the sample covariance matrix
is estimated from 1,200 observations for 210 covariances — poor, but survivable, which is
why the index-model covariance does not dominate it here. Push n toward 500, as slide 5
does, and the sample matrix stops being invertible long before it stops being noisy. The
factor structure is what makes the problem well-posed at all.

## Export

Bundle the figures and the tables, in case you want them for the transfer questions or
your own notes.

In [ ]:
# TODO: export the three figures you produced together with the summary tables

## Where this goes next

Two quizzes are open in Moodle until Sunday: the cumulative drill and the transfer
questions. Both are ungraded, and both are exactly the format the exams use.

Task 3 left one question open. The index model tells you how much to bet on a view; it has
nothing to say about where the view comes from, because everything it knows is already in
past returns. Lecture 09 asks a related question from the other side — if α is zero for
everyone, what does the cross-section of β have to look like? — and turns the regression
you ran twenty times today into a test of an equilibrium model.